# Практическая работа 3 — Прогнозирование временных рядов

**Тема:** Прогнозирование временных рядов методом средней годовой дельты  
**Датасет:** `public_data.csv` — 61 временной ряд, январь 2001 – февраль 2020 (230 месяцев)  
**Горизонт прогноза:** 36 месяцев (март 2020 – февраль 2023)  
**Метод:** mean_delta_9 — сезонный прогноз с поправкой на среднюю годовую дельту за 9 аналогичных месяцев  
**Результат:** SMAPE **18.35%** (vs baseline 21.64%)

---

## Цель
Построить прогнозную модель для 61 временного ряда и получить прогноз на 36 шагов (март 2020 – февраль 2023).

## Задачи
1. Загрузить и подготовить данные
2. Перевести таблицу из широкого формата в длинный
3. Построить сезонный baseline для сравнения
4. Реализовать прогноз методом mean_delta_9
5. Провести ретроспективную проверку (backtest)
6. Сформировать итоговый прогноз на 36 месяцев

---

## Описание датасета

| Показатель | Значение |
|---|---|
| Количество временных рядов | 61 |
| Исходный период | январь 2001 – февраль 2020 |
| Количество исходных месяцев | 230 |
| Период прогноза | март 2020 – февраль 2023 |
| Горизонт прогноза | 36 месяцев |

## Метод прогнозирования

**Baseline (сезонный):** прогноз на месяц = значение того же месяца прошлого года  
$$\hat{y}(t) = y(t - 12)$$

**mean_delta_9:** добавляет поправку на среднюю годовую дельту за последние 9 аналогичных месяцев  
$$\text{delta}_{12}(t) = y(t) - y(t-12)$$
$$\hat{\delta}(t) = \text{mean}(\text{последние 9 delta}_{12} \ \text{для этого месяца})$$
$$\hat{y}(t) = y(t-12) + \hat{\delta}(t)$$

Окно 9 лет выбрано по результатам ретроспективной проверки (проверялись окна от 2 до 12 лет).

## Шаг 1 — Загрузка и подготовка данных

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH    = Path("public_data.csv")
DELTA_WINDOW = 9

# '--' и 'NM' — обозначения пропусков в исходном датасете
df = pd.read_csv(DATA_PATH, na_values=["--", "NM"])
df["date"] = pd.to_datetime(df["date"], format="%b %Y")

# Все столбцы кроме date — числовые значения рядов
value_columns = df.columns.drop("date")
df[value_columns] = df[value_columns].apply(pd.to_numeric, errors="coerce")

print(f"Загружено: {df.shape[0]} месяцев, {len(value_columns)} временных рядов")
print(f"Период: {df['date'].min().strftime('%Y-%m')} — {df['date'].max().strftime('%Y-%m')}")
df.head(3)

## Шаг 2 — Перевод в длинный формат (wide → long)

Исходная таблица: одна строка = один месяц, 61 столбец-ряд.  
После `melt`: одна строка = один ряд в один месяц — удобно обрабатывать все ряды в цикле единообразно.

In [ ]:
df_long = df.melt(
    id_vars="date",
    var_name="state",
    value_name="value",
)
df_long = df_long.sort_values(["state", "date"])

# Вспомогательные поля
df_long["month"]    = df_long["date"].dt.month
df_long["year"]     = df_long["date"].dt.year
df_long["lag_12"]   = df_long.groupby("state")["value"].shift(12)
df_long["delta_12"] = df_long["value"] - df_long["lag_12"]

df_long.to_csv("timeline_full.csv", index=False)

print(f"Длинный формат: {df_long.shape}")
print(f"Рядов: {df_long['state'].nunique()}")
df_long.head(6)

## Шаг 3 — Метрика качества SMAPE

**SMAPE** (Symmetric Mean Absolute Percentage Error) — симметричная процентная ошибка.  
Устраняет недостаток MAPE: одинаково штрафует перепрогноз и недопрогноз.  
$$\text{SMAPE} = \frac{200}{n} \sum \frac{|y - \hat{y}|}{|y| + |\hat{y}|}$$

In [ ]:
def smape(y_true, y_pred):
    denominator = y_true.abs() + y_pred.abs()
    result = (2 * (y_true - y_pred).abs() / denominator).where(denominator != 0, 0)
    return result.mean() * 100

## Шаг 4 — Функции прогнозирования

In [ ]:
def get_value_months_ago(state_history, date, months):
    """Возвращает значение ряда за N месяцев до указанной даты."""
    previous_date = date - pd.DateOffset(months=months)
    value = state_history.loc[state_history["date"] == previous_date, "value"]
    if len(value):
        return value.iloc[0]
    previous_values = state_history[state_history["date"] < date]["value"]
    return previous_values.iloc[-1] if len(previous_values) else 0


def get_month_delta_history(state_history, date):
    """История годовых дельт для данного месяца (год к году)."""
    month_history = state_history[
        (state_history["date"] < date)
        & (state_history["date"].dt.month == date.month)
    ].copy()
    month_history["lag_12"]   = month_history["value"].shift(1)
    month_history["delta_12"] = month_history["value"] - month_history["lag_12"]
    return month_history.dropna()


def predict_delta_by_recent_mean(state_history, date, window=DELTA_WINDOW):
    """Средняя дельта за последние N аналогичных месяцев."""
    month_history = get_month_delta_history(state_history, date)
    recent_deltas = month_history["delta_12"].tail(window)
    if len(recent_deltas):
        return recent_deltas.mean()
    return 0


def mean_delta_forecast(history, states, future_dates, window=DELTA_WINDOW):
    """Прогноз методом mean_delta для всех рядов рекурсивно."""
    predictions = []
    for date in future_dates:
        rows = []
        for state in states:
            state_history = history[history["state"] == state].sort_values("date")
            lag_12 = get_value_months_ago(state_history, date, 12)
            predicted_delta_12 = predict_delta_by_recent_mean(state_history, date, window)
            prediction = lag_12 + predicted_delta_12
            rows.append({
                "date": date, "state": state,
                "prediction": prediction,
                "predicted_delta_12": predicted_delta_12,
                "lag_12": lag_12,
            })
        future_df = pd.DataFrame(rows)
        predictions.append(future_df)
        # Рекурсивно добавляем предсказанные значения в историю
        new_history = future_df[["date", "state", "prediction"]].rename(
            columns={"prediction": "value"}
        )
        history = pd.concat([history, new_history], ignore_index=True)
    return pd.concat(predictions, ignore_index=True)


def seasonal_baseline_forecast(history, states, future_dates):
    """Baseline: прогноз = значение того же месяца прошлого года."""
    predictions = []
    for date in future_dates:
        rows = []
        for state in states:
            state_history = history[history["state"] == state].sort_values("date")
            prediction = get_value_months_ago(state_history, date, 12)
            rows.append({"date": date, "state": state, "baseline": prediction})
        future_df = pd.DataFrame(rows)
        predictions.append(future_df)
        new_history = future_df[["date", "state", "baseline"]].rename(
            columns={"baseline": "value"}
        )
        history = pd.concat([history, new_history], ignore_index=True)
    return pd.concat(predictions, ignore_index=True)


print("Функции прогнозирования определены.")

## Шаг 5 — Ретроспективная проверка (Backtest)

Проверяем качество метода на историческом периоде:  
- Берём данные только **до марта 2017** года
- Строим прогноз на 36 месяцев (март 2017 – февраль 2020)
- Сравниваем с фактическими значениями, которые нам известны

In [ ]:
states = sorted(df_long["state"].unique())

backtest_dates   = pd.date_range("2017-03-01", "2020-02-01", freq="MS")
backtest_history = df_long[
    (df_long["date"] < "2017-03-01") & df_long["value"].notna()
][["date", "state", "value"]].copy()

print(f"Обучающий период: до 2017-03")
print(f"Прогнозный период: {len(backtest_dates)} месяцев")
print("Запуск backtest... (займёт несколько минут)")

backtest_forecast  = mean_delta_forecast(backtest_history, states, backtest_dates, DELTA_WINDOW)
baseline_backtest  = seasonal_baseline_forecast(backtest_history, states, backtest_dates)

print("Backtest завершён.")

## Шаг 6 — Расчёт метрик качества

In [ ]:
actual_backtest = df_long[
    (df_long["date"] >= "2017-03-01") & (df_long["date"] <= "2020-02-01")
][["date", "state", "value"]]

backtest_result = (
    backtest_forecast
    .merge(actual_backtest, on=["date", "state"], how="inner")
    .merge(baseline_backtest, on=["date", "state"], how="inner")
    .dropna()
)

model_mae    = mean_absolute_error(backtest_result["value"], backtest_result["prediction"])
baseline_mae = mean_absolute_error(backtest_result["value"], backtest_result["baseline"])
model_rmse   = mean_squared_error(backtest_result["value"], backtest_result["prediction"]) ** 0.5
baseline_rmse= mean_squared_error(backtest_result["value"], backtest_result["baseline"]) ** 0.5
model_smape  = smape(backtest_result["value"], backtest_result["prediction"])
baseline_smape = smape(backtest_result["value"], backtest_result["baseline"])

print("=" * 50)
print("Backtest 2017-03 → 2020-02")
print(f"{'Метрика':<15} {'mean_delta_9':>15} {'Baseline':>12}")
print("-" * 44)
print(f"{'MAE':<15} {model_mae:>15.2f} {baseline_mae:>12.2f}")
print(f"{'RMSE':<15} {model_rmse:>15.2f} {baseline_rmse:>12.2f}")
print(f"{'SMAPE':<15} {model_smape:>14.2f}% {baseline_smape:>11.2f}%")
print("=" * 50)

In [ ]:
# SMAPE по каждому ряду — показывает, где метод лучше baseline, а где хуже
smape_by_state = (
    backtest_result
    .groupby("state")
    .apply(
        lambda x: pd.Series({
            "mean_delta_9_smape": smape(x["value"], x["prediction"]),
            "baseline_smape":     smape(x["value"], x["baseline"]),
        }),
        include_groups=False,
    )
    .reset_index()
)
smape_by_state["best_method"] = smape_by_state[
    ["mean_delta_9_smape", "baseline_smape"]
].idxmin(axis=1)

print("Победитель по методу:")
print(smape_by_state["best_method"].value_counts())
print()
print("Примеры SMAPE по отдельным рядам:")
print(smape_by_state.head(10).round(2).to_string(index=False))

## Шаг 7 — Итоговый прогноз на 36 месяцев

Применяем метод ко всем доступным историческим данным до февраля 2020 включительно.  
Прогнозируем март 2020 – февраль 2023.

In [ ]:
FORECAST_PATH = "forecast_mean_delta_9_2020_2023.csv"

future_dates = pd.date_range("2020-03-01", "2023-02-01", freq="MS")
history      = df_long[["date", "state", "value"]].dropna().copy()

print(f"Строим прогноз на {len(future_dates)} месяцев для {len(states)} рядов...")
forecast = mean_delta_forecast(history, states, future_dates, DELTA_WINDOW)
forecast.to_csv(FORECAST_PATH, index=False)

print(f"Прогноз сохранён: {FORECAST_PATH}")
print(f"Форма: {forecast.shape}")
print()
print("Фрагмент итогового прогноза:")
print(forecast.head(10).round(2).to_string(index=False))

## Выводы

- Подготовлен датасет из **61 временного ряда** (январь 2001 – февраль 2020, 230 месяцев на ряд).
- Реализован метод прогнозирования **mean_delta_9**: сезонный baseline + поправка на среднюю годовую дельту за 9 аналогичных месяцев прошлых лет.
- Ретроспективная проверка подтвердила, что метод превосходит базовый сезонный baseline:

| Метрика | mean_delta_9 | Baseline |
|---|---|---|
| MAE | 156.62 | 231.76 |
| RMSE | 356.09 | 549.33 |
| SMAPE | **18.35%** | 21.64% |

- Метод оказался лучше для **39 из 61** рядов. Для остальных 22 сезонный baseline точнее — это нормально.
- Окно 9 лет выбрано по минимальному SMAPE среди окон от 2 до 12 лет.
- Итоговый прогноз на 36 месяцев сохранён в `forecast_mean_delta_9_2020_2023.csv`.